<style>
.mermaid {
  width: 100%;
  overflow-x: auto;
  padding: 1.25rem 0 1.75rem;
}
.mermaid svg {
  width: 100% !important;
  max-width: 1120px !important;
  min-width: 0 !important;
  height: auto !important;
  display: block;
  margin: 0 auto;
}
</style>

# Task 6. Idempotent Replay Verification

## Mục tiêu

Phần Idempotent Replay Verification xác minh và chứng minh cơ chế replay và tính idempotent cho nhánh đồ thị từ Kafka sang Neo4j. Task 5 độc lập kiểm chứng cơ chế khôi phục checkpoint của Spark và ghi đè MongoDB. Trong luồng hoàn chỉnh, Parser Service sinh DELETE/UPSERT events từ graph diff, Neo4j Kafka Sink áp dụng mutation idempotent theo stable ID, và Spark Structured Streaming upsert metadata vào MongoDB theo `file_id`.

Task 6 verifies replay and idempotency for the Kafka-to-Neo4j graph path. Task 5 independently verifies Spark checkpoint recovery and MongoDB upsert behavior.

## Tiêu chí xác minh

| Tiêu chí | Bằng chứng trong pipeline | Trạng thái |
|---|---|---|
| File không đổi được bỏ qua | `ProcessFileService` so sánh `content_hash`, `parser_version`, `schema_version`. | Verified |
| File đã sửa sinh replay events | `ReplayFileService` parse lại, gọi `CpgDiffer`, rồi publish DELETE/UPSERT events. | Verified |
| Không dùng ID ngẫu nhiên | `IdentifierGenerator` sinh `file_id`, `node_id`, `edge_id`, `event_id` bằng SHA-256 deterministic. | Verified |
| State được commit sau publish thành công | `ProcessFileService` validate và flush events trước khi commit SQLite. | Verified |
| Neo4j không duplicate | Kafka Sink dùng uniqueness constraints, Cypher `MERGE` và tombstone; bằng chứng runtime nằm ở Task 4. | Verified |
| MongoDB metadata upsert | Spark writer dùng `replace`/`upsertDocument=true` theo `file_id`; bằng chứng runtime nằm ở Task 5. | Verified |
| Checkpoint không đọc lại offset cũ | Spark Structured Streaming duy trì `checkpointLocation` cho topic `source.metadata`; bằng chứng runtime nằm ở Task 5. | Verified |

## Thiết kế replay idempotent

```mermaid
%%{init: {
  "theme": "base",
  "themeVariables": {
    "fontFamily": "Inter, Arial, sans-serif",
    "fontSize": "22px",
    "primaryColor": "#eef2ff",
    "primaryBorderColor": "#4f46e5",
    "primaryTextColor": "#111827",
    "lineColor": "#334155",
    "clusterBkg": "#f8fafc",
    "clusterBorder": "#cbd5e1"
  },
  "flowchart": {
    "htmlLabels": true,
    "nodeSpacing": 58,
    "rankSpacing": 76,
    "curve": "basis"
  }
}}%%
flowchart TB
    Old["State cũ<br/>hash v1 + IDs v1"]
    Modified["File đã sửa"]
    Parser["Parser Service"]
    Diff["Graph diff"]

    Old --> Diff
    Modified --> Parser --> Diff

    Diff --> Delete["DELETE events<br/>dọn phần tử cũ"]
    Diff --> Upsert["UPSERT events<br/>ghi graph mới"]
    Diff --> Meta["Metadata event<br/>thống kê file"]

    Delete --> Kafka["Kafka"]
    Upsert --> Kafka
    Meta --> Kafka

    Kafka --> Neo4j["Neo4j<br/>MERGE theo stable ID"]
    Kafka --> Spark["Spark Streaming<br/>checkpoint offsets"]
    Spark --> Mongo["MongoDB<br/>upsert by file_id"]
    Parser -->|"sau khi publish + flush thành công"| NewState["SQLite state mới<br/>hash v2 + IDs v2"]
```

## Chuẩn bị notebook runtime

Quy trình xác minh dùng một file fixture riêng trong `workspace/tmp/task6-replay-verification/source` để không sửa repository mục tiêu thật. Cách làm này vẫn đi qua đúng service của dự án: `GitSourceRepository`, `CpgParser`, `ProcessFileService`, `ReplayFileService`, `SqliteStateStore`, `EventValidator` và `JsonlEventWriter`.

In [1]:
from pathlib import Path
import json
import shutil
import sqlite3
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "lab04-book":
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from application.services.process_file import ProcessFileService
from application.services.replay_file import ReplayFileService
from domain.models import SourceFile
from infrastructure.filesystem.git_source_repository import GitSourceRepository
from infrastructure.messaging.event_validator import EventValidator
from infrastructure.messaging.jsonl_event_writer import JsonlEventWriter
from infrastructure.state.sqlite_state_store import SqliteStateStore
from parsing.cpg_parser import CpgParser
from parsing.identifiers import IdentifierGenerator

verification_dir = PROJECT_ROOT / "workspace" / "tmp" / "task6-replay-verification"
source_root = verification_dir / "source"
output_dir = verification_dir / "events"
state_db = verification_dir / "parser_state.sqlite3"
repo_id = "local/task6-fixture"
target_file = Path("sample_module.py")

if verification_dir.exists():
    shutil.rmtree(verification_dir)
source_root.mkdir(parents=True)
output_dir.mkdir(parents=True)
(source_root / target_file).write_text(
    """def total(values):
    result = 0
    for value in values:
        result += value
    return result
""",
    encoding="utf-8",
)

repo = GitSourceRepository(source_root, "", None)
parser = CpgParser(repository_id=repo_id)
state_store = SqliteStateStore(state_db, repo_id)
validator = EventValidator(PROJECT_ROOT / "schemas")
writer = JsonlEventWriter(output_dir)
process_service = ProcessFileService(repo, parser, state_store, validator, writer)
replay_service = ReplayFileService(repo, parser, state_store, process_service, repo_id)

source_file = SourceFile(
    repo_id,
    str(source_root),
    target_file.as_posix(),
    "local-fixture",
    (source_root / target_file).stat().st_size,
)
first_result = process_service.execute(source_file)
unchanged_result = process_service.execute(source_file)

print("first_status=", first_result.status.value)
print("unchanged_status=", unchanged_result.status.value)
print("first_nodes=", first_result.node_count)
print("first_edges=", first_result.edge_count)

first_status= SUCCESS
unchanged_status= SKIPPED_UNCHANGED
first_nodes= 26
first_edges= 33


## Thực thi: replay sau khi file thay đổi

File fixture được sửa bằng cách thêm một nhánh điều kiện và thay đổi logic trả về. Replay phải phát hiện `content_hash` mới, tính diff với trạng thái cũ, phát hành các event cần thiết và cập nhật SQLite state sau khi writer flush thành công.

In [2]:
(source_root / target_file).write_text(
    """def total(values):
    result = 0
    for value in values:
        if value is None:
            continue
        result += value
    return result * 2
""",
    encoding="utf-8",
)

replay_result = replay_service.execute(target_file)
print(json.dumps(replay_result, indent=2, ensure_ascii=False))

{
  "file_path": "sample_module.py",
  "status": "SUCCESS",
  "old_content_hash": "cfa7852de016d35f44f92b1a00eff9c3493944d345e2d88fee4c0dacda021ad2",
  "new_content_hash": "97759d2fffc1db5cee11f265408fccaa036f44f4ab655602fa2e25c84b2e3c76",
  "removed_node_count": 8,
  "removed_edge_count": 14,
  "upsert_node_count": 36,
  "upsert_edge_count": 45,
  "error": null
}


## Xác minh: event và SQLite state

Phần này đếm các event JSONL đã sinh ra trong cả hai lần xử lý, tương đương payload sẽ được publish lên Kafka trong live mode. Kỳ vọng quan trọng là lần chạy không đổi không làm tăng event graph, còn lần replay có DELETE/UPSERT events để downstream Neo4j và MongoDB cập nhật đúng bản ghi hiện có.

In [3]:
def read_jsonl(path: Path) -> list[dict]:
    if not path.exists():
        return []
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]

nodes = read_jsonl(output_dir / "nodes.jsonl")
edges = read_jsonl(output_dir / "edges.jsonl")
metadata = read_jsonl(output_dir / "metadata.jsonl")

event_type_counts = {}
for event in [*nodes, *edges, *metadata]:
    event_type_counts[event["event_type"]] = event_type_counts.get(event["event_type"], 0) + 1

file_id = IdentifierGenerator.generate_file_id(repo_id, target_file)
with sqlite3.connect(state_db) as conn:
    row = conn.execute(
        "SELECT file_id, file_path, content_hash, node_ids_json, edge_ids_json FROM file_state WHERE file_id = ?",
        (file_id,),
    ).fetchone()

state_summary = {
    "event_type_counts": event_type_counts,
    "metadata_event_count": len(metadata),
    "state_file_id_matches": row[0] == file_id,
    "state_file_path": row[1],
    "state_content_hash": row[2],
    "state_node_count": len(json.loads(row[3])),
    "state_edge_count": len(json.loads(row[4])),
}
print(json.dumps(state_summary, indent=2, ensure_ascii=False))

{
  "event_type_counts": {
    "NODE_UPSERT": 62,
    "NODE_DELETE": 8,
    "EDGE_UPSERT": 78,
    "EDGE_DELETE": 14,
    "FILE_METADATA_UPSERT": 2
  },
  "metadata_event_count": 2,
  "state_file_id_matches": true,
  "state_file_path": "sample_module.py",
  "state_content_hash": "97759d2fffc1db5cee11f265408fccaa036f44f4ab655602fa2e25c84b2e3c76",
  "state_node_count": 36,
  "state_edge_count": 45
}


## Kết quả

- Lần đầu tiên trả về `SUCCESS`, chứng minh parser có thể tạo graph và commit state ban đầu.
- Lần chạy ngay sau đó trả về `SKIPPED_UNCHANGED`, chứng minh cơ chế incremental không parse lại file có cùng hash.
- Sau khi sửa file, replay trả về `SUCCESS`, `old_content_hash` khác `new_content_hash`, đồng thời `removed_node_count` và `removed_edge_count` thể hiện diff giữa graph cũ và graph mới.
- SQLite chỉ giữ một bản ghi theo `file_id`, cho thấy state cục bộ được overwrite thay vì tạo state trùng lặp.
- Neo4j nhận node/edge events qua Kafka Connect Sink và ghi bằng `MERGE`, nên replay cùng stable IDs chỉ cập nhật graph hiện có thay vì tạo duplicate.
- MongoDB được cập nhật bằng upsert theo `file_id`, vì vậy metadata sau replay phản ánh phiên bản mới nhất của file.

## Lệnh xác minh end-to-end

```powershell
docker compose --env-file .env -f infra/docker-compose.yml up -d kafka neo4j kafka-connect mongodb
./scripts/create_topics.sh
./scripts/register_connectors.sh
uv run lab04 replay-file --file path/to/modified.py --no-dry-run
cypher-shell -u neo4j -p $env:NEO4J_PASSWORD -f scripts/verify_neo4j.cypher
mongosh $env:MONGODB_URI scripts/verify_mongodb.js
powershell -ExecutionPolicy Bypass -File scripts/run_metadata_to_mongodb.ps1 -AvailableNow
```


## Reflection

Idempotent replay chỉ hoàn chỉnh khi cả producer, message broker và downstream storage cùng tuân thủ stable identity. Parser Service đã cung cấp deterministic IDs, skip unchanged files, graph diff và commit SQLite sau publish; Neo4j Sink hoàn tất vòng lặp bằng `MERGE`/delete idempotent; Spark và MongoDB hoàn tất nhánh metadata bằng checkpoint và upsert theo `file_id`. Nhờ vậy, replay một file đã sửa cập nhật trạng thái mới mà không làm tăng số node, edge hoặc metadata document một cách sai lệch.